# Pitch benchmark

**November 11 volume promotion / Run All:** Live Attune now uses 0.04 × running p95 RMS (including silence and the current frame); post-hoc retains 0.04 × whole-recording p95. Confidence thresholds are unchanged. Restart the kernel before Run All.

`attune_only_refresh=True` locks every competitor to existing suite rows. Missing or incompatible competitor results stop execution before detection; no competitor is rerun. Only stale Attune rows refresh, and later runs reuse them. Benchmark pitch cache v16 and Attune-only streaming checkpoint identities ensure the new defaults are evaluated without invalidating app/demo caches. All offline and online pairwise cells recompute from the returned updated rows.

Preliminary and expanded 449-track suites run by default; optional full suites remain disabled. Historical notes below describe earlier promotions.

**`pyin`** is the local librosa-equivalent baseline; **`attune`** calls the production pYIN+ detector and smoother. Other methods retain their saved estimates and scores.


### Fair competitor configuration

Every external estimate is now given the same reference/score-derived admissible F0 range. APIs that support range-aware candidate selection (for example Attune, Praat, and Basic Pitch) use it internally; fixed-output APIs are also range-gated after inference. This distinction matters for SwiftF0: its [official API](https://github.com/lars76/swift-f0/) says the network always detects over 46.875–2093.75 Hz and `fmin`/`fmax` only decide which outputs count as voiced. The shared final gate therefore supplies the same information without claiming that it changes SwiftF0's CNN inference. Praat remains raw AC because Praat documents it as the periodicity method; its filtered recommendation is aimed at speech intonation and vocal-fold analysis.

SPICE uses one fixed confidence threshold of **0.35** across every dataset and the streaming adapter. Predictions below it remain unvoiced for scoring. The threshold was chosen after the exploratory confidence sweep; these preliminary results are not a held-out estimate of threshold selection performance.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "app.py").is_file()
    )
)
sys.path.insert(0, str(REPO_ROOT))
import importlib

for module_name in (
    "algorithms.PitchSmoother",
    "benchmarks.modules.pitch.PitchCache",
    "benchmarks.modules.pitch.competitors.Attune",
    "benchmarks.modules.pitch.competitors.Spice",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.datasets.Bach10",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.PitchNotebook",
):
    importlib.reload(importlib.import_module(module_name))
from benchmarks.modules.pitch.PitchNotebook import PitchNotebook


In [3]:
PRIMARY_METHODS = ("pyin", "attune", "praat", "spice")
RUN_FULL_RUNS = False
CONFIG = PitchNotebook.NotebookConfig(use_cache=True, force_rerun=False, attune_only_refresh=True)
BENCH = PitchNotebook(CONFIG)
CACHE_METHODS = tuple(BENCH.methods)
BENCH.show_configuration()
RUN_PAPER_EVALUATION = True
OFFLINE_METHODS = tuple(BENCH.methods)
ONLINE_METHODS = ("attune", "pyin_framewise", "praat", "swiftf0", "spice", "mcleod", "crepe_tiny")
PAPER_ONLINE_METHODS = ONLINE_METHODS
PAIRED_RESAMPLES = 9999


,setting,value
0,workers,8
1,streaming workers,8
2,preliminary tracks per instrument,2
3,full CocoChorales stems per stratum,all
4,random seed,0
5,methods,"pyin, attune, basic_pitch, praat, swiftf0, cre..."
6,estimate caches,reused
7,degradation SNRs,"clean, 20dB, 15dB, 10dB, 5dB"
8,run cache,/Users/sarah/Desktop/shasha/Attune/benchmarks/...


## Method availability

Third-party trackers are optional dependencies, and a missing one is skipped rather than fatal. Run this first so a long cell later does not quietly drop a method: anything marked `MISSING` carries the install hint that would make it available. `torchcrepe` and `penn` are excluded by default because they stack a second heavy PyTorch runtime; set `include_slow_methods=True` on the config to include them, ideally in their own invocation.

In [4]:
availability = BENCH.show_availability()


,method,status,description,detail
0,pyin,OK,Local librosa-compatible pYIN at Attune's 44.1...,
1,attune,OK,Attune pYIN+ promoted production pipeline,
2,basic_pitch,OK,"Basic Pitch contour head, projected to one sub...",
3,praat,OK,Praat autocorrelation (praat-parselmouth),
4,swiftf0,OK,SwiftF0 CNN (swift-f0),
5,crepe,OK,"CREPE CNN, TensorFlow reference implementation",
6,spice,OK,SPICE self-supervised pitch estimator (TF Hub),
7,rmvpe,OK,RMVPE deep U-Net (vendored),
8,mcleod,OK,"Tartini 1.2 native MPM (stateful, official sou...",


## Preliminary sampling check

Every preliminary suite uses the same sampling contract: `CONFIG.preliminary_tracks_per_instrument` tracks for every instrument label, selected independently within each label using `CONFIG.seed`. With the defaults this yields 8 original Bach10 isolated recordings (4 instruments × 2), 26 URMP stems (13 × 2), and 26 CocoChorales stems (13 × 2). The check below also confirms that the same seed repeats exactly while the next seed changes track identities without changing those quotas.

In [5]:
preliminary_sampling = BENCH.show_preliminary_sampling()


,seed,instruments,tracks per instrument,total tracks,same seed repeats,next seed changes tracks
corpus,,,,,,
Bach10 original,0,4,2,8,True,True
URMP,0,13,2,26,True,True
CocoChorales,0,13,2,26,True,True


## Preliminary comparison — original Bach10 recordings

Two seed-selected recordings from each of Bach10's four instruments (violin, clarinet, saxophone, bassoon; 8 total). Only isolated monophonic stems are evaluated; ensemble mixes are excluded. Each method's search range is tightened to the reference melody's range.

These are real performances, replacing `bach10-mf0-synth` in this suite. The supplied `*-GTF0s.mat` annotations contain fractional MIDI pitches: the loader converts them to Hz, preserves unvoiced frames, and uses the documented 10 ms hop with the first frame centered at 23 ms. These references were produced with YIN and manual corrections, so they are not independent physical measurements of F0. See the [official description](https://labsites.rochester.edu/air/datasets/Bach10%20Dataset_v1.0.pdf).

Data location: `benchmarks/datasets/bach10-original/<piece>/`. The local v1.1 release comes from this [public mirror, pinned revision](https://github.com/flippy-fyp/Bach10_v1.1/tree/2a53cdc6495bb82f03f943b77ca3b11ddf7f5a31). To provision another checkout, extract that archive into the data location, retaining the ten piece directories. Missing original data raises an error; synthesized audio is never substituted.

This cell runs all configured available methods, then resumes cached results on subsequent runs. Original results use `preliminary_bach10_original`; historical synthesized results under `preliminary` remain unchanged.


In [6]:
preliminary = BENCH.run_preliminary(cache_methods=CACHE_METHODS)


Reusing completed run: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/preliminary_bach10_original/seeded_per_instrument_v1__per_instrument_2__seed_0__methods_pyin+attune+basic_pitch+praat+swiftf0+crepe+spice+rmvpe+mcleod


,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
pyin,8,0.9709,0.9715,0.9214,0.9799,0.4644,3.2271
attune,8,0.9596,0.9602,0.9287,0.9671,0.3121,8.2137
basic_pitch,8,0.9210,0.9223,0.8958,0.9635,0.3005,42.2087
praat,8,0.9579,0.9629,0.9312,0.9787,0.2763,273.7342
swiftf0,8,0.9251,0.9252,0.8913,0.9370,0.3726,42.1260
crepe,8,0.9717,0.9723,0.9302,0.9809,0.3930,0.1887
spice,8,0.8972,0.8987,0.8567,0.9320,0.4597,18.7602
rmvpe,8,0.9477,0.9506,0.9134,0.9676,0.3537,5.5586
mcleod,8,0.9222,0.9247,0.8843,0.9482,0.4116,266.1234


## Preliminary URMP comparison

Run the same detector set on two seed-selected isolated stems from each of URMP's 13 instrument classes (26 total). The adapter pairs each `AuSep` WAV with its 10 ms `F0s` annotation and carries the piece, ensemble size, instrument, and part number into the result rows. This is the quickest balanced check that the downloaded corpus and every installed method work end to end.

In [7]:
preliminary_urmp = BENCH.run_preliminary_urmp(force=False, cache_methods=CACHE_METHODS)


Reusing completed run: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/preliminary_urmp/seeded_per_instrument_v1__per_instrument_2__seed_0__methods_pyin+attune+basic_pitch+praat+swiftf0+crepe+spice+rmvpe+mcleod__attune_pyin_plus_ordinary__live_unv_0p985_volume_0p04__posthoc_unv_0p9875_volume_0p04


,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
pyin,26,0.9581,0.9644,0.8553,0.9862,0.4831,3.7193
attune,26,0.9455,0.9516,0.9328,0.9699,0.1087,8.2887
basic_pitch,26,0.9079,0.9110,0.8815,0.9788,0.2054,40.8921
praat,26,0.9329,0.9435,0.9275,0.9582,0.0902,276.6245
swiftf0,26,0.8535,0.8567,0.8440,0.8750,0.1873,34.0198
crepe,26,0.9305,0.9342,0.9210,0.9467,0.1102,0.1798
spice,26,0.8890,0.8917,0.8539,0.9467,0.2618,31.3096
rmvpe,26,0.9229,0.9413,0.8842,0.9693,0.2432,6.2521
mcleod,26,0.9410,0.9532,0.8988,0.9743,0.2400,274.3531


### Low-latency real-time replay

This live comparison has one controlled pair, one explicitly frame-restarted local pYIN diagnostic, and two neural adapters. Attune's ordinary-candidate frame-local pYIN with the selected causal `0.985` confidence and `0.04 × running-p95 RMS` gates and raw Praat receive the identical 44.1-kHz capture frame, hop, center timestamp, and score-derived nominal pitch range. `pyin_framewise` runs the local implementation on each newly available 4096-sample, left-aligned frame (`center=False`) at 44.1 kHz. It uses the reference single-frame joint voiced/unvoiced state decision, without an extra 0.5 probability gate or temporal smoothing; it remains distinct from the full-track `pyin` baseline.

SwiftF0 and SPICE use their smallest practical model-native rolling center frames; every update commits one estimate only after its required future samples arrive. SwiftF0 uses 320 ms of receptive context / about 152 ms look-ahead at 16 ms updates, SPICE 64 ms / 32 ms at 32 ms updates. RMVPE is excluded from online runs because its pilot configuration accumulated backlog; it remains in the offline comparison. There is no whole-file call, 1--2 second bounded-history rerun, or timestamp ablation in this table. Model loading and one warm-up call happen before steady-state timing.

The first table uses the notebook's standard pitch-summary columns. The second retains per-recording interaction diagnostics with the same title-case naming convention, and the third shows each adapter's native geometry while making the Attune/Praat match explicit and the frame-restarted librosa geometry visible. Attune uses a `0.04` running-p95 RMS floor in live mode and a `0.04` whole-track p95 RMS floor post hoc. Pass `instruments=()` to sample one stem from every selected corpus instrument, and set `force=True` only when intentionally replacing otherwise current rows.

**Latency retention fix (2026-10-04):** v7 saved only per-recording percentiles, so its exact pooled P95 cannot be recovered. v9 saves every output-update latency, CPU duration, and estimate timestamp in compressed checkpoint sidecars. Ordinary reruns now use a new versioned folder and rerun all methods once; completed v9 recordings are reused only while their latency samples remain valid. Existing v7 results are preserved. Restart the kernel before rerunning after this code update.

Methods use up to all but two logical CPUs (`CONFIG.streaming_workers` sets the ceiling), with concurrency reduced according to available RAM before each method, reusing warmed models within each worker. Progress shows the latest completed track with an elapsed timer. Each completed song/method is checkpointed immediately; rerun with `force=False` to resume, or `force=True` to recompute. Frames remain sequential. Like `pyin_ablation.ipynb`, output latency and deadline misses use dedicated-worker process CPU simulation; measured wall compute time remains a separate diagnostic and can increase under contention. The timing cache version separates these results from the older wall-latency rows.

**pYIN correction (2026-10-04):** v9 removes the nonstandard 50% voiced-probability cutoff. The single-frame decision compares the strongest voiced state with one unvoiced state, matching reference one-frame Viterbi with uniform initialization. Restart the kernel and rerun the online cells; `force=False` uses fresh v9 output folders and preserves older results. Previous streaming pYIN accuracy and paired comparisons must be replaced.


The scheduler leaves two logical CPUs free and reserves the larger of 2 GiB or 15% of physical RAM for the desktop. Each method starts with one calibration job, then budgets measured peak worker memory plus 30% (at least 0.5 GiB per worker). Before a real detection has completed, unmeasured heavy methods retain a 2 GiB budget; cached predictions do not lower it. Offline jobs contain at most four tracks (one for CREPE). CREPE divides the CPU budget across its TensorFlow inference pools: with an eight-core budget, one RAM-limited worker can use eight threads, or two workers can use four threads each. Other methods retain one numerical thread per worker. New CREPE caches record `inference_threads`; older caches remain reusable and retain their original timing. CPU timing still sums process CPU time across threads, so it is not wall throughput. Streaming pools reuse warmed models for batches of eight tracks per worker. Pools release their memory and recheck available RAM between batches. If available RAM falls below the desktop reserve at a batch boundary, execution stops with a resumable error. These are scheduling estimates, not hard memory limits.

**McLeod / Tartini 1.2:** `mcleod` now executes the official C++ MPM engine with its default 2048-sample window and 1024-sample hop at 44.1 kHz. It retains persistent filtering, noise-floor scoring, note tracking, warped NSDF aggregation and retrospective octave correction. Streaming commits causal estimates with 23.22 ms look-ahead; offline reads revised history on the native grid. The common score-range gate applies after inference. See `benchmarks/modules/pitch/competitors/tartini/README.md` for the pinned source, build requirements, explicit fixes, and why the later thesis pipeline differs from the linked release. Timing excludes GUI/device I/O.


In [8]:
streaming_urmp = BENCH.run_streaming_comparison(
    instruments=(), tracks_per_instrument=2, force=False, suite_methods=ONLINE_METHODS
)


Running low-latency live replay: 26 excerpt(s), attune
Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[  12/26] attune: 11/26 finished | 0s | 11 cached; loading model  attune: 8 worker(s); forced 8-worker policy (limited only by remaining tracks)
[  26/26] attune: 25/26 finished | 274s | finished: 34_Fugue_tpt_tpt_hn_tbn/AuSep_3_hn_34_Fugue         
Report: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/streaming_urmp/native_center_v9_pyin_joint_state_latency_samples__frame_counts_10ms_v1__methods_attune+pyin_framewise+praat+swiftf0+spice+mcleod+crepe_tiny__instruments_all__per_instrument_2__seed_0__max_infs__unv_0p985__dynamic_floor_0p03/rows.csv


,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
attune,26,0.9427,0.9505,0.9288,0.9703,0.1169,6.7126
pyin_framewise,26,0.9603,0.9689,0.7709,0.9994,0.8525,3.6804
praat,26,0.9268,0.9507,0.8358,0.9986,0.4640,2.1521
swiftf0,26,0.8495,0.8526,0.8431,0.8704,0.1781,1.8485
spice,26,0.8272,0.8418,0.8067,0.8991,0.2608,7.6095
mcleod,26,0.9391,0.9520,0.8983,0.9759,0.2359,214.8201
crepe_tiny,26,0.8415,0.8497,0.8353,0.9034,0.1851,34.5338


Transition VFA  \
model      Track ID                                                             
attune     02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                        0.3689   
           05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer           0.4969   
           08_Spring_fl_vn/AuSep_1_fl_08_Spring                        0.4464   
           12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                     0.5796   
           14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                       0.5813   
...                                                                       ...   
crepe_tiny 41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere          0.5119   
           42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso           0.5688   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Ch...          0.5520   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Ch...          0.6336   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Ch...          0.4290   

                                                               Deep-rest VFA  \
model      Track ID                                                            
attune     02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                       0.0000   
           05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer          0.0000   
           08_Spring_fl_vn/AuSep_1_fl_08_Spring                       0.0000   
           12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                    0.0000   
           14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                      0.0000   
...                                                                      ...   
crepe_tiny 41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere         0.0000   
           42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso          0.0000   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Ch...         0.0000   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Ch...         0.0000   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Ch...         0.0000   

                                                               Median Onset Acquisition (ms)  \
model      Track ID                                                                            
attune     02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                                       0.0000   
           05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer                          0.0000   
           08_Spring_fl_vn/AuSep_1_fl_08_Spring                                       0.0000   
           12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                                    0.0000   
           14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                                      0.0000   
...                                                                                      ...   
crepe_tiny 41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere                        55.0000   
           42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso                         20.0000   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Ch...                        30.0000   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Ch...                         0.0000   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Ch...                        60.0000   

                                                               Median Release Hangover (ms)  \
model      Track ID                                                                           
attune     02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                                      0.0000   
           05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer                        20.0000   
           08_Spring_fl_vn/AuSep_1_fl_08_Spring                                     20.0000   
           12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                                  10.0000   
           14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                                    20.0000   
...                                                   

Instrument  \
model      Track ID                                                        
attune     02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                   violin   
           05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer     trumpet   
           08_Spring_fl_vn/AuSep_1_fl_08_Spring                    flute   
           12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                violin   
           14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                   flute   
...                                                                  ...   
crepe_tiny 41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere    bassoon   
           42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso        horn   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Ch...    trumpet   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Ch...   trombone   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Ch...       tuba   

                                                                                           Stream Adapter  \
model      Track ID                                                                                         
attune     02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                                         Matched app frame   
           05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer                            Matched app frame   
           08_Spring_fl_vn/AuSep_1_fl_08_Spring                                         Matched app frame   
           12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                                      Matched app frame   
           14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                                        Matched app frame   
...                                                                                                   ...   
crepe_tiny 41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere  CREPE tiny, web demo callback (Python CPU)   
           42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso   CREPE tiny, web demo callback (Python CPU)   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Ch...  CREPE tiny, web demo callback (Python CPU)   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Ch...  CREPE tiny, web demo callback (Python CPU)   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Ch...  CREPE tiny, web demo callback (Python CPU)   

                                                               Frame Size (samples)  \
model      Track ID                                                                   
attune     02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                                4096   
           05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer                   4096   
           08_Spring_fl_vn/AuSep_1_fl_08_Spring                                4096   
           12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring                             4096   
           14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                               4096   
...                                                                             ...   
crepe_tiny 41_Miserere_fl_fl_ob_sax_bn/AuSep_5_bn_41_Miserere                  4096   
           42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso                   4096   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_2_tpt_43_Ch...                  4096   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Ch...                  4096   
           43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Ch...                  4096   

                                                               Integration Size (samples)  \
model      Track ID                                                                         
attune     02_Sonata_vn_vn/AuSep_2_vn_02_Sonata                                 4096.0000   
           05_Entertainer_tpt_tpt/AuSep_2_tpt_05_Entertainer                    4096.0000   
           08_Spring_fl_vn/AuSep_1_fl_08_Spring                                 4096.0000   
           12_Spring_vn_vn_vc/AuSep_1_vn_12_Spring        

## Preliminary CocoChorales comparison

The same methods over two seed-selected stems from each of CocoChorales' 13 instrument classes (26 total). The helper prefers two different source compositions within each class and materializes the selected stems automatically. Rows carry ensemble/instrument metadata, and each search range is taken from a robust percentile of the annotated f0 rather than its extremes.

In [9]:
preliminary_coco = BENCH.run_preliminary_coco(force=False, cache_methods=CACHE_METHODS)


materialized 0 file(s) for test
Running: preliminary_coco (refreshing stale attune)
materialized 0 file(s) for test
  attune: 0/26 cached predictions available for rescoring; only cache misses or incompatible configurations run detection
Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  26/26] attune: test__woodwind_track228859__2_oboe     
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/preliminary_coco/seeded_per_instrument_v1__per_instrument_2__seed_0__methods_pyin+attune+basic_pitch+praat+swiftf0+crepe+spice+rmvpe+mcleod


,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
pyin,26,0.9648,0.9673,0.9500,0.9868,0.3191,6.1414
attune,26,0.9612,0.9636,0.9557,0.9798,0.1434,7.8474
basic_pitch,26,0.9658,0.9663,0.9462,0.9959,0.4086,37.5741
praat,26,0.9474,0.9575,0.9423,0.9836,0.1491,679.2098
swiftf0,26,0.9057,0.9062,0.8954,0.9276,0.2911,38.2504
crepe,26,0.9617,0.9633,0.9507,0.9832,0.2489,0.1825
spice,26,0.9352,0.9363,0.9215,0.9722,0.3283,23.1450
rmvpe,26,0.9260,0.9270,0.9202,0.9680,0.1848,5.0990
mcleod,26,0.8479,0.8515,0.8516,0.8710,0.0797,267.0145


## Offline paired analysis

Pool all five metrics across the eight Bach10, 26 URMP, and 26 CocoChorales preliminary tracks. Compare Attune with each offline competitor on pooled overall accuracy using paired source-piece label swaps and 95% paired cluster-bootstrap intervals. Apply Holm correction within the offline family. Every comparison and the headline table use the same completed track intersection; coverage and excluded tracks are reported. Missing methods raise an error rather than silently changing the test family.

Frames retain equal scoring weight, while dependent frames and stems stay together for inference. `source_groups` can link known shared compositions across corpora. Intervals are unadjusted; adjusted p-values determine significance. These development samples do not establish held-out generalization.


In [10]:
offline_paired = BENCH.show_paired_analysis(
    preliminary,
    preliminary_urmp,
    preliminary_coco,
    mode="Offline",
    methods=OFFLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES,
)


Offline: frame-pooled scores on the common completed recording set; Holm correction within this mode. Confidence intervals are unadjusted.


,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,8,8,27955,24777,3178
coco,26,26,53986,51162,2824
urmp,26,14,236769,181591,55178


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9607,0.9656,0.8771,0.9857,0.4745
attune,0.9500,0.9549,0.9364,0.9716,0.1209
basic_pitch,0.9207,0.9231,0.8937,0.9807,0.2197
praat,0.9382,0.9482,0.9304,0.9652,0.1026
swiftf0,0.8708,0.8731,0.8569,0.8914,0.2017
crepe,0.9407,0.9436,0.9268,0.9572,0.1313
spice,0.8990,0.9013,0.8656,0.9504,0.2751
rmvpe,0.9259,0.9393,0.8928,0.9689,0.2462
mcleod,0.9207,0.9302,0.8896,0.9513,0.2415


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin,Overall Accuracy,60,48,0,318710,0,9999,5.9207,3.4420,8.6399,0.0001,9999,0.0008,True
1,basic_pitch,Overall Accuracy,60,48,0,318710,0,9999,4.2612,2.6735,5.9953,0.0001,9999,0.0008,True
2,praat,Overall Accuracy,60,48,0,318710,0,9999,0.5987,-0.1289,1.5434,0.1645,9999,0.1645,False
3,swiftf0,Overall Accuracy,60,48,0,318710,0,9999,7.9492,5.0758,12.5970,0.0001,9999,0.0008,True
4,crepe,Overall Accuracy,60,48,0,318710,0,9999,0.9504,0.2762,1.7421,0.0173,9999,0.0346,True
5,spice,Overall Accuracy,60,48,0,318710,0,9999,7.0798,4.8387,10.0678,0.0001,9999,0.0008,True
6,rmvpe,Overall Accuracy,60,48,0,318710,0,9999,4.3522,2.7957,6.2623,0.0001,9999,0.0008,True
7,mcleod,Overall Accuracy,60,48,0,318710,0,9999,4.6789,3.2064,6.2846,0.0001,9999,0.0008,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/offline/cc0dc05b9692


## Online paired analysis

Add the same eight Bach10 and 26 CocoChorales tracks to the 26-track URMP streaming replay. Pool all five metrics across datasets, then compare overall accuracy, voicing recall, and voicing false-alarm rate against each of the five online competitors. Holm correction covers these fifteen online tests separately from offline tests. Source-piece pairing, confidence intervals, coverage checks, and frame weights follow the offline analysis. The streaming pYIN adaptation is included in all three paired comparisons.


In [11]:
streaming_bach10 = BENCH.run_streaming_comparison(
    dataset="bach10-original", instruments=(), tracks_per_instrument=2, force=False, suite_methods=ONLINE_METHODS
)
streaming_coco = BENCH.run_streaming_comparison(
    dataset="coco", instruments=(), tracks_per_instrument=2, force=False, suite_methods=ONLINE_METHODS
)
online_paired = BENCH.show_paired_analysis(
    streaming_bach10,
    streaming_urmp,
    streaming_coco,
    mode="Streaming",
    methods=ONLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES,
)


Running low-latency live replay: 8 excerpt(s), attune
Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[   1/8] attune: 0/8 finished | 0s | 0 cached; loading model  attune: 8 worker(s); forced 8-worker policy (limited only by remaining tracks)
[   8/8] attune: 7/8 finished | 37s | finished: 02-AchLiebenChristen-violin
Report: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/streaming_bach10-original/native_center_v9_pyin_joint_state_latency_samples__frame_counts_10ms_v1__methods_attune+pyin_framewise+praat+swiftf0+spice+mcleod+crepe_tiny__instruments_all__per_instrument_2__seed_0__max_infs__unv_0p985__dynamic_floor_0p03/rows.csv


,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
attune,8,0.9583,0.9594,0.9267,0.9687,0.3194,6.8121
pyin_framewise,8,0.9754,0.9769,0.8826,0.9975,0.8408,4.6980
praat,8,0.9687,0.9713,0.8971,0.9954,0.6611,2.3465
swiftf0,8,0.9234,0.9235,0.8912,0.9353,0.3603,2.0353
spice,8,0.8447,0.8578,0.7950,0.8884,0.5928,8.7558
mcleod,8,0.9297,0.9310,0.8908,0.9512,0.4125,243.3698
crepe_tiny,8,0.9227,0.9228,0.8835,0.9701,0.4216,36.9084


Transition VFA  Deep-rest VFA  \
model          Track ID                                                       
attune         01-AchGottundHerr-violin               0.8804         0.2329   
               02-AchLiebenChristen-violin            0.8734         0.0000   
               04-ChristeDuBeistand-clarinet          0.7328         0.2938   
               05-DieNacht-clarinet                   0.3830         0.0000   
               06-DieSonne-bassoon                    0.5158         0.0000   
               07-HerrGott-saxphone                   0.7460         0.0000   
               08-FuerDeinenThron-bassoon             0.7222         0.1024   
               10-NunBitten-saxphone                  0.5568         0.0156   
pyin_framewise 01-AchGottundHerr-violin               1.0000         0.9452   
               02-AchLiebenChristen-violin            1.0000         0.9800   
               04-ChristeDuBeistand-clarinet          0.9542         0.6959   
               05-DieNacht-clarinet                   0.7979         0.6881   
               06-DieSonne-bassoon                    0.9895         0.8218   
               07-HerrGott-saxphone                   0.8889         0.5922   
               08-FuerDeinenThron-bassoon             1.0000         0.8855   
               10-NunBitten-saxphone                  0.8750         0.7969   
praat          01-AchGottundHerr-violin               1.0000         0.9315   
               02-AchLiebenChristen-violin            0.9937         0.9800   
               04-ChristeDuBeistand-clarinet          0.9084         0.3557   
               05-DieNacht-clarinet                   0.5957         0.2569   
               06-DieSonne-bassoon                    0.9579         0.5644   
               07-HerrGott-saxphone                   0.9365         0.2330   
               08-FuerDeinenThron-bassoon             0.9815         0.6084   
               10-NunBitten-saxphone                  0.7727         0.3438   
swiftf0        01-AchGottundHerr-violin               0.8478         0.8082   
               02-AchLiebenChristen-violin            0.6203         0.7000   
               04-ChristeDuBeistand-clarinet          0.6260         0.3557   
               05-DieNacht-clarinet                   0.4043         0.0000   
               06-DieSonne-bassoon                    0.5789         0.0693   
               07-HerrGott-saxphone                   0.6667         0.0000   
               08-FuerDeinenThron-bassoon             0.5926         0.2169   
               10-NunBitten-saxphone                  0.3977         0.0156   
spice          01-AchGottundHerr-violin               1.0000         0.5616   
               02-AchLiebenChristen-violin            0.9747         0.8800   
               04-ChristeDuBeistand-clarinet          0.7863         0.6186   
               05-DieNacht-clarinet                   0.5851         0.3761   
               06-DieSonne-bassoon                    0.3263         0.2772   
               07-HerrGott-saxphone                   0.9048         0.4563   
               08-FuerDeinenThron-bassoon             0.6296         0.2711   
               10-NunBitten-saxphone                  0.7273         0.5000   
mcleod         01-AchGottundHerr-violin               0.9565         0.6575   
               02-AchLiebenChristen-violin            0.9114         0.6600   
               04-ChristeDuBeistand-clarinet          0.7634         0.3454   
               05-DieNacht-clarinet                   0.3830         0.0459   
               06-DieSonne-bassoon                    0.5368         0.0396   
               07-HerrGott-saxphone                   0.6825         0.0000   
               08-FuerDeinenThron-bassoon             0.7037         0.1867   
               10-NunBitten-saxphone                  0.4886         0.0156   
crepe_tiny     01-AchGottundHerr-violin               0.9130         0.6027   
               02

Instrument  \
model          Track ID                                   
attune         01-AchGottundHerr-violin          violin   
               02-AchLiebenChristen-violin       violin   
               04-ChristeDuBeistand-clarinet   clarinet   
               05-DieNacht-clarinet            clarinet   
               06-DieSonne-bassoon              bassoon   
               07-HerrGott-saxphone           saxophone   
               08-FuerDeinenThron-bassoon       bassoon   
               10-NunBitten-saxphone          saxophone   
pyin_framewise 01-AchGottundHerr-violin          violin   
               02-AchLiebenChristen-violin       violin   
               04-ChristeDuBeistand-clarinet   clarinet   
               05-DieNacht-clarinet            clarinet   
               06-DieSonne-bassoon              bassoon   
               07-HerrGott-saxphone           saxophone   
               08-FuerDeinenThron-bassoon       bassoon   
               10-NunBitten-saxphone          saxophone   
praat          01-AchGottundHerr-violin          violin   
               02-AchLiebenChristen-violin       violin   
               04-ChristeDuBeistand-clarinet   clarinet   
               05-DieNacht-clarinet            clarinet   
               06-DieSonne-bassoon              bassoon   
               07-HerrGott-saxphone           saxophone   
               08-FuerDeinenThron-bassoon       bassoon   
               10-NunBitten-saxphone          saxophone   
swiftf0        01-AchGottundHerr-violin          violin   
               02-AchLiebenChristen-violin       violin   
               04-ChristeDuBeistand-clarinet   clarinet   
               05-DieNacht-clarinet            clarinet   
               06-DieSonne-bassoon              bassoon   
               07-HerrGott-saxphone           saxophone   
               08-FuerDeinenThron-bassoon       bassoon   
               10-NunBitten-saxphone          saxophone   
spice          01-AchGottundHerr-violin          violin   
               02-AchLiebenChristen-violin       violin   
               04-ChristeDuBeistand-clarinet   clarinet   
               05-DieNacht-clarinet            clarinet   
               06-DieSonne-bassoon              bassoon   
               07-HerrGott-saxphone           saxophone   
               08-FuerDeinenThron-bassoon       bassoon   
               10-NunBitten-saxphone          saxophone   
mcleod         01-AchGottundHerr-violin          violin   
               02-AchLiebenChristen-violin       violin   
               04-ChristeDuBeistand-clarinet   clarinet   
               05-DieNacht-clarinet            clarinet   
               06-DieSonne-bassoon              bassoon   
               07-HerrGott-saxphone           saxophone   
               08-FuerDeinenThron-bassoon       bassoon   
               10-NunBitten-saxphone          saxophone   
crepe_tiny     01-AchGottundHerr-violin          violin   
               02-AchLiebenChristen-violin       violin   
               04-ChristeDuBeistand-clarinet   clarinet   
               05-DieNacht-clarinet            clarinet   
               06-DieSonne-bassoon              bassoon   
               07-HerrGott-saxphone           saxophone   
               08-FuerDeinenThron-bassoon       bassoon   
               10-NunBitten-saxphone          saxophone   

                                                                                 Stream Adapter  \
model          Track ID                                                                           
attune         01-AchGottundHerr-violin                                       Matched app frame   
               02-AchLiebenChristen-violin                                    Matched app frame   
               04-ChristeDuBeistand-clarinet                                  Matched app frame   
               05-DieNacht-clarinet                                           Matched app frame   
               06-DieSonne-b

materialized 0 file(s) for test
materialized 0 file(s) for test
Running low-latency live replay: 26 excerpt(s), attune
Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[   1/26] attune: 0/26 finished | 0s | 0 cached; loading model  attune: 8 worker(s); forced 8-worker policy (limited only by remaining tracks)
[  26/26] attune: 25/26 finished | 65s | finished: test__woodwind_track229162__4_bassoon  
Report: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/streaming_coco/native_center_v9_pyin_joint_state_latency_samples__frame_counts_10ms_v1__methods_attune+pyin_framewise+praat+swiftf0+spice+mcleod+crepe_tiny__instruments_all__per_instrument_2__seed_0__max_infs__unv_0p985__dynamic_floor_0p03/rows.csv


,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
attune,26,0.9599,0.9641,0.9545,0.9818,0.1434,6.8834
pyin_framewise,26,0.9673,0.9723,0.9214,0.9996,0.9097,5.0714
praat,26,0.9367,0.9557,0.9018,0.9984,0.7309,2.5524
swiftf0,26,0.9041,0.9046,0.8955,0.9259,0.2610,2.0344
spice,26,0.9138,0.9155,0.8948,0.9550,0.4494,8.6646
mcleod,26,0.8425,0.8464,0.8468,0.8663,0.0765,238.3379
crepe_tiny,26,0.9080,0.9089,0.8947,0.9584,0.3467,36.5378


... (4 outputs left out)

,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,8,8,27955,24777,3178
coco,26,26,53986,51162,2824
urmp,26,14,236768,181591,55177


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
attune,0.9476,0.9540,0.9330,0.9724,0.1287
pyin_framewise,0.9631,0.9703,0.8062,0.9993,0.8545
praat,0.9328,0.9537,0.8523,0.9982,0.4866
swiftf0,0.8675,0.8697,0.8562,0.8877,0.1914
spice,0.8461,0.8580,0.8206,0.9092,0.2867
mcleod,0.9190,0.9290,0.8889,0.9518,0.2377
crepe_tiny,0.8625,0.8685,0.8496,0.9207,0.2049


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin_framewise,Overall Accuracy,60,48,0,318709,0,9999,12.6802,9.2293,16.0661,0.0001,9999,0.0018,True
1,pyin_framewise,Voicing Recall,60,48,0,257530,0,9999,-2.6882,-3.9664,-1.6025,0.0001,9999,0.0018,True
2,pyin_framewise,Voicing False Alarm,60,48,0,61179,0,9999,-72.5837,-77.0641,-66.4886,0.0001,9999,0.0018,True
3,praat,Overall Accuracy,60,48,0,318709,0,9999,8.0632,6.5258,9.6526,0.0001,9999,0.0018,True
4,praat,Voicing Recall,60,48,0,257530,0,9999,-2.5838,-3.8067,-1.5461,0.0001,9999,0.0018,True
5,praat,Voicing False Alarm,60,48,0,61179,0,9999,-35.7917,-47.4479,-25.9932,0.0001,9999,0.0018,True
6,swiftf0,Overall Accuracy,60,48,0,318709,0,9999,7.6791,4.7806,12.2515,0.0001,9999,0.0018,True
7,swiftf0,Voicing Recall,60,48,0,257530,0,9999,8.4728,4.1900,14.7075,0.0001,9999,0.0018,True
8,swiftf0,Voicing False Alarm,60,48,0,61179,0,9999,-6.2685,-13.3796,-0.4815,0.0587,9999,0.0587,False
9,spice,Overall Accuracy,60,48,0,318709,0,9999,11.2356,7.6418,15.7386,0.0001,9999,0.0018,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/streaming/fa8b3a8f13f6


## Expanded 449-track evaluation

The offline cell retains the existing evaluation. To run only the updated online experiment, skip the offline cell and use **Expanded online native-geometry experiment** below: first its dry-run cell, then its execution cell. Both require the setup cells only.

The selection remains 40 Bach10, 149 URMP and 260 CocoChorales (20 per instrument), with existing reference alignment, pitch ranges and scoring tolerances. The new comparison uses the same development benchmark set; it is not new held-out validation.


In [12]:
paper_offline = BENCH.run_paper_evaluation("Offline")
paper_offline_paired = BENCH.show_paired_analysis(
    *paper_offline,
    mode="Offline",
    methods=OFFLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES
)

Running: paper_offline_bach10-original (refreshing stale attune)
  attune: 8/40 cached predictions available for rescoring; only cache misses or incompatible configurations run detection
Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  40/40] attune: 10-NunBitten-violin                    
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paper_offline_bach10-original/seed_0__per_instrument_None


,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
pyin,40,0.9733,0.9742,0.9258,0.9844,0.4519,3.9530
attune,40,0.9678,0.9687,0.9357,0.9769,0.3198,7.4199
basic_pitch,40,0.9254,0.9263,0.8965,0.9692,0.3333,42.1262
praat,40,0.9620,0.9640,0.9363,0.9810,0.2680,275.3249
swiftf0,40,0.9426,0.9429,0.9041,0.9557,0.4021,42.0183
crepe,40,0.9742,0.9750,0.9330,0.9838,0.3944,0.1891
spice,40,0.8929,0.8940,0.8531,0.9367,0.4637,25.1765
rmvpe,40,0.9628,0.9640,0.9270,0.9823,0.3573,5.7627
mcleod,40,0.9428,0.9444,0.9023,0.9647,0.4191,266.7357


Running: paper_offline_urmp (refreshing stale attune)
  attune: 26/149 cached predictions available for rescoring; only cache misses or incompatible configurations run detection
Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  64/149] attune: 25_Pirates_vn_vn_va_sax/AuSep_2_vn_25_Pirates          Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 128/149] attune: 39_Jerusalem_vn_vn_va_sax_db/AuSep_3_va_39_Jerusalem   Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 149/149] attune: 44_K515_vn_vn_va_va_vc/AuSep_5_vc_44_K515              
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paper_offline_urmp/seed_0__per_instrument_None


,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
pyin,149,0.9580,0.9678,0.8677,0.9862,0.4181,3.8446
attune,149,0.9459,0.9555,0.9309,0.9709,0.1167,6.6825
basic_pitch,149,0.9221,0.9266,0.8946,0.9821,0.1926,44.3905
praat,149,0.9280,0.9417,0.9293,0.9543,0.0664,312.0452
swiftf0,149,0.8376,0.8420,0.8340,0.8599,0.1775,44.1094
crepe,149,0.9411,0.9476,0.9294,0.9603,0.1075,0.1922
spice,149,0.9014,0.9072,0.8731,0.9579,0.2163,35.8333
rmvpe,149,0.9222,0.9456,0.8888,0.9690,0.2171,8.9448
mcleod,149,0.9472,0.9633,0.9071,0.9797,0.2198,279.8208


materialized 0 file(s) for test
Running: paper_offline_coco (refreshing stale attune)
materialized 0 file(s) for test
  attune: 26/260 cached predictions available for rescoring; only cache misses or incompatible configurations run detection
Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  64/260] attune: test__brass_track223823__4_tuba    Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 128/260] attune: test__random_track235273__1_flute      Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 192/260] attune: test__string_track216994__4_cello      Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 256/260] attune: test__woodwind_track229821__3_clarinet Pitch batch (attune): 4/8 worker(s) × 1 inference thread(s); force

,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,,,,,,,
pyin,260,0.9677,0.9705,0.9520,0.9884,0.3288,7.2723
attune,260,0.9613,0.9640,0.9550,0.9796,0.1571,7.8486
basic_pitch,260,0.9676,0.9685,0.9461,0.9972,0.4390,42.7552
praat,260,0.9489,0.9571,0.9429,0.9809,0.1637,698.6951
swiftf0,260,0.9156,0.9163,0.9036,0.9379,0.3098,45.9029
crepe,260,0.9652,0.9663,0.9524,0.9847,0.2769,0.1924
spice,260,0.9367,0.9383,0.9227,0.9757,0.3269,33.0180
rmvpe,260,0.9304,0.9319,0.9242,0.9701,0.1873,9.7695
mcleod,260,0.8604,0.8639,0.8624,0.8803,0.1019,268.2340


Offline: frame-pooled scores on the common completed recording set; Holm correction within this mode. Confidence intervals are unadjusted.


,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,40,10,133688,118748,14940
coco,260,256,556720,527274,29446
urmp,149,27,1654312,1257159,397153


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9616,0.9689,0.8910,0.9867,0.4133
attune,0.9515,0.9587,0.9369,0.9737,0.1262
basic_pitch,0.9349,0.9382,0.9069,0.9855,0.2138
praat,0.9359,0.9473,0.9329,0.9633,0.0797
swiftf0,0.8657,0.8689,0.8545,0.8875,0.1939
crepe,0.9498,0.9545,0.9351,0.9685,0.1285
spice,0.9106,0.9150,0.8838,0.9615,0.2320
rmvpe,0.9270,0.9430,0.8994,0.9701,0.2198
mcleod,0.9229,0.9346,0.8962,0.9512,0.2187


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin,Overall Accuracy,449,293,0,2344720,0,9999,4.5857,3.1701,5.7282,0.0001,9999,0.0008,True
1,basic_pitch,Overall Accuracy,449,293,0,2344720,0,9999,2.9945,2.2845,3.8356,0.0001,9999,0.0008,True
2,praat,Overall Accuracy,449,293,0,2344720,0,9999,0.3937,-0.0114,0.8982,0.0588,9999,0.1176,False
3,swiftf0,Overall Accuracy,449,293,0,2344720,0,9999,8.2381,6.6823,9.7874,0.0001,9999,0.0008,True
4,crepe,Overall Accuracy,449,293,0,2344720,0,9999,0.1806,-0.1057,0.4905,0.2447,9999,0.2447,False
5,spice,Overall Accuracy,449,293,0,2344720,0,9999,5.3114,4.6421,6.0405,0.0001,9999,0.0008,True
6,rmvpe,Overall Accuracy,449,293,0,2344720,0,9999,3.7511,2.6680,4.7490,0.0001,9999,0.0008,True
7,mcleod,Overall Accuracy,449,293,0,2344720,0,9999,4.0660,3.1191,5.2213,0.0001,9999,0.0008,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/offline/7e08c0e75d49


### Expanded online native-geometry experiment (November 2026)

Run the setup cells, then the dry-run and execution cells below to run only the same **449 tracks (40 Bach10, 149 URMP, 260 CocoChorales)**. This explicit selective policy supersedes `attune_only_refresh` for this experiment. Keep `force_rerun=False`. Historical online and all offline results remain separate.

Only pYIN (4096/1024 samples at 44.1 kHz; joint frame decision without sequence HMM), Praat (native one-frame raw AC, 3/floor window and 0.75/floor hop, no cross-frame tracking), and CREPE-tiny (64 ms windows / 10 ms updates, no Viterbi) get new inference identities. Praat uses the shared range directly, without the app wrapper's padding or 40 Hz clamp; its silence reference is buffer-local. Attune running-p95, SwiftF0, SPICE and native Tartini must reuse compatible checkpoints. Their missing or incompatible evidence blocks the run **before any inference**. Neural models retain their fixed inference ranges and common final output range gate.

The dry-run saves exact per-track misses and reasons. Execution repeats preflight, resumes completed method/track checkpoints, and returns the new accuracy and **P95 feedback latency (ms)** table. This is the duration-weighted steady-state age of the latest available estimate, including held outputs, analysis look-ahead and simulated process-CPU computation/queueing. Startup and device/rendering delays are excluded; playback ends at audio duration without a drain period. Timing coverage and startup diagnostics, per-track geometry, derived metric caches and provenance are saved alongside the report. Paired tests below are recomputed from the returned rows. This is a configuration comparison on the already-used set, not held-out validation.


In [4]:
native_feedback_plan = BENCH.run_native_feedback_evaluation(dry_run=True)


materialized 0 file(s) for test


,reusable_predictions,reusable_timing_sidecars,reporting_only,inference_miss,blocked
model,,,,,
attune,449,449,449,0,0
pyin_framewise,189,189,189,260,0
praat,189,189,189,260,0
swiftf0,449,449,449,0,0
spice,449,449,449,0,0
mcleod,449,449,449,0,0
crepe_tiny,124,124,124,325,0


Exact method/track plan: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/native_feedback_2026_11_v1/dry_run_plan.csv (845 inference misses)
  crepe_tiny: 325 track(s): no checkpoint for this adapter identity
  pyin_framewise: 260 track(s): no checkpoint for this adapter identity
  praat: 260 track(s): no checkpoint for this adapter identity


In [5]:
# Expanded 449 online experiment only: run setup, the dry-run cell above, then this cell.
# Explicit opt-in by executing this cell; offline/preliminary runs are unnecessary.
paper_online, paper_online_table = BENCH.run_native_feedback_evaluation(dry_run=False)
paper_online_paired = BENCH.show_paired_analysis(
    *paper_online,
    mode="Streaming",
    methods=PAPER_ONLINE_METHODS,
)
paper_online_table


materialized 0 file(s) for test


,reusable_predictions,reusable_timing_sidecars,reporting_only,inference_miss,blocked
model,,,,,
attune,449,449,449,0,0
pyin_framewise,189,189,189,260,0
praat,189,189,189,260,0
swiftf0,449,449,449,0,0
spice,449,449,449,0,0
mcleod,449,449,449,0,0
crepe_tiny,124,124,124,325,0


Exact method/track plan: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/native_feedback_2026_11_v1/dry_run_plan.csv (845 inference misses)
  crepe_tiny: 325 track(s): no checkpoint for this adapter identity
  pyin_framewise: 260 track(s): no checkpoint for this adapter identity
  praat: 260 track(s): no checkpoint for this adapter identity
Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[   1/65] crepe_tiny: 0/65 finished | 0s | 0 cached; loading model  crepe_tiny: 1 worker(s); 2.00 GiB/worker budget
[   2/65] crepe_tiny: 1/65 finished | 92s | finished: 30_Fugue_fl_fl_ob_sax/AuSep_3_ob_30_Fugue  crepe_tiny: 1 worker(s); 0.88 GiB/worker budget
[  10/65] crepe_tiny: 9/65 finished | 597s | finished: 32_Fugue_vn_vn_va_vc/AuSep_3_va_32_Fugue            crepe_tiny: 2 worker(s); 0.88 GiB/worker budget
[  26/65] crepe_tiny: 25/65 finished | 1108s | finished: 36_Rondeau_vn_vn_va_vc/AuSep_3_va_36_Rondeau      crepe_tiny: 

,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,recordings,timing_coverage_failures,evaluated_seconds,evaluated_duration_coverage,P95 feedback latency (ms)
model,,,,,,,,,,
attune,0.9498,0.9588,0.9336,0.9756,0.1363,449,0,23405.8412,0.9982,49.5724
crepe_tiny,0.9226,0.9368,0.9139,0.9533,0.1238,449,0,23417.6219,0.9987,43.9959
mcleod,0.9198,0.9320,0.8944,0.9506,0.2152,449,0,23437.2771,0.9996,45.3726
praat,0.9537,0.9640,0.9140,0.9941,0.2567,449,0,23440.1156,0.9997,27.6264
pyin_framewise,0.9593,0.9681,0.8195,0.9994,0.7831,449,0,23405.7712,0.9982,69.0366
spice,0.8820,0.8944,0.8525,0.9388,0.2745,449,0,23417.2208,0.9987,66.0958
swiftf0,0.8624,0.8654,0.8536,0.8835,0.1842,449,0,23300.4273,0.9937,175.1768


P95 feedback latency is the time-weighted 95th percentile of the age of the latest available pitch estimate during playback, measured from its audio timestamp. It includes analysis look-ahead, simulated computation and queueing, and time between updates. Initial startup and screen-rendering delay are excluded.


,recordings,output_updates,pooled_p95_output_latency_ms
model,,,
attune,449,8064346,47.0049
crepe_tiny,449,2342172,34.7928
mcleod,449,1009585,23.3350
praat,449,6702990,15.4685
pyin_framewise,449,1008238,47.1319
spice,449,732102,36.0000
swiftf0,449,1456761,160.0832


... (2 outputs left out)

,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
attune,0.9498,0.9588,0.9336,0.9756,0.1363
pyin_framewise,0.9593,0.9681,0.8195,0.9994,0.7831
praat,0.9537,0.9640,0.9140,0.9941,0.2567
swiftf0,0.8624,0.8654,0.8536,0.8835,0.1842
spice,0.8820,0.8944,0.8525,0.9388,0.2745
mcleod,0.9198,0.9320,0.8944,0.9506,0.2152
crepe_tiny,0.9226,0.9368,0.9139,0.9533,0.1238


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin_framewise,Overall Accuracy,449,293,0,2344717,0,9999,11.4053,9.0034,13.3207,0.0001,9999,0.0018,True
1,pyin_framewise,Voicing Recall,449,293,0,1903181,0,9999,-2.3795,-3.2656,-1.7570,0.0001,9999,0.0018,True
2,pyin_framewise,Voicing False Alarm,449,293,0,441536,0,9999,-64.6858,-70.7199,-59.5123,0.0001,9999,0.0018,True
3,praat,Overall Accuracy,449,293,0,2344717,0,9999,1.9519,1.4898,2.4919,0.0001,9999,0.0018,True
4,praat,Voicing Recall,449,293,0,1903181,0,9999,-1.8450,-2.6411,-1.2833,0.0001,9999,0.0018,True
5,praat,Voicing False Alarm,449,293,0,441536,0,9999,-12.0416,-16.4439,-8.7743,0.0001,9999,0.0018,True
6,swiftf0,Overall Accuracy,449,293,0,2344717,0,9999,7.9956,6.4887,9.4510,0.0001,9999,0.0018,True
7,swiftf0,Voicing Recall,449,293,0,1903181,0,9999,9.2095,7.0692,11.1193,0.0001,9999,0.0018,True
8,swiftf0,Voicing False Alarm,449,293,0,441536,0,9999,-4.7962,-9.6164,-0.9560,0.0024,9999,0.0048,True
9,spice,Overall Accuracy,449,293,0,2344717,0,9999,8.1051,7.1183,9.2190,0.0001,9999,0.0018,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/streaming/c7cafe12f7c3


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,recordings,timing_coverage_failures,evaluated_seconds,evaluated_duration_coverage,P95 feedback latency (ms)
model,,,,,,,,,,
attune,0.9498,0.9588,0.9336,0.9756,0.1363,449,0,23405.8412,0.9982,49.5724
crepe_tiny,0.9226,0.9368,0.9139,0.9533,0.1238,449,0,23417.6219,0.9987,43.9959
mcleod,0.9198,0.9320,0.8944,0.9506,0.2152,449,0,23437.2771,0.9996,45.3726
praat,0.9537,0.9640,0.9140,0.9941,0.2567,449,0,23440.1156,0.9997,27.6264
pyin_framewise,0.9593,0.9681,0.8195,0.9994,0.7831,449,0,23405.7712,0.9982,69.0366
spice,0.8820,0.8944,0.8525,0.9388,0.2745,449,0,23417.2208,0.9987,66.0958
swiftf0,0.8624,0.8654,0.8536,0.8835,0.1842,449,0,23300.4273,0.9937,175.1768


## Interpreting raw pitch accuracy

Raw pitch accuracy uses a 50-cent tolerance; raw chroma accuracy additionally ignores octave errors. Under these adapters' zero-frequency unvoiced convention, raw pitch accuracy decreases both when played tones are suppressed and when voiced estimates have the wrong pitch. The following decomposition uses pooled reference-voiced frame counts. It is not a mean of track-level percentages. Negative-frequency latent predictions, if introduced by another adapter, would require revisiting this decomposition.


In [14]:
rpa_decomposition = BENCH.show_rpa_decomposition(
    preliminary, preliminary_urmp, preliminary_coco, methods=PRIMARY_METHODS
)


Tracks  Raw Pitch Accuracy  \
Suite                       Method                               
preliminary_bach10_original pyin         8              0.9709   
                            attune       8              0.9596   
                            praat        8              0.9579   
                            spice        8              0.8972   
preliminary_urmp            pyin        26              0.9581   
                            attune      26              0.9455   
                            praat       26              0.9329   
                            spice       26              0.8890   
preliminary_coco            pyin        26              0.9648   
                            attune      26              0.9612   
                            praat       26              0.9474   
                            spice       26              0.9352   

                                    Voicing Recall  Missed-voicing share  \
Suite                       Method                                         
preliminary_bach10_original pyin            0.9799                0.0201   
                            attune          0.9671                0.0329   
                            praat           0.9787                0.0213   
                            spice           0.9320                0.0680   
preliminary_urmp            pyin            0.9862                0.0138   
                            attune          0.9699                0.0301   
                            praat           0.9582                0.0418   
                            spice           0.9467                0.0533   
preliminary_coco            pyin            0.9868                0.0132   
                            attune          0.9798                0.0202   
                            praat           0.9836                0.0164   
                            spice           0.9722                0.0278   

                                    Wrong-pitch share  RPA / Voicing Recall  
Suite                       Method                                           
preliminary_bach10_original pyin               0.0090                0.9908  
                            attune             0.0075                0.9922  
                            praat              0.0209                0.9787  
                            spice              0.0348                0.9627  
preliminary_urmp            pyin               0.0281                0.9715  
                            attune             0.0244                0.9748  
                            praat              0.0252                0.9737  
                            spice              0.0578                0.9390  
preliminary_coco            pyin               0.0219                0.9778  
                            attune             0.0186                0.9810  
                            praat              0.0363                0.9631  
                            spice              0.0369                0.9620

## Full comparisons

These cells drop the preliminary per-instrument sampler and run every track in the configured corpora, including all 40 original Bach10 isolated recordings and all 149 isolated real-recording URMP stems. They are long and cache-backed: jobs run in up to `CONFIG.workers` spawned workers with one numerical-library thread per worker, except CREPE, which shares that CPU budget between processes and TensorFlow inference threads, and any method whose dependency is missing is skipped rather than failing the run.

**Skipped by Run All by default.** Set `RUN_FULL_RUNS = True` in the configuration cell to enable these suites.
The full audio suite contains original Bach10 plus the two MDB synthesized corpora. Its new `all_audio_original_bach10` result directory preserves the previous `all_audio` results.


In [15]:
if RUN_FULL_RUNS:
    all_audio = BENCH.run_all_audio()
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


Skipped full run (RUN_FULL_RUNS=False).


In [16]:
if RUN_FULL_RUNS:
    all_urmp = BENCH.run_all_urmp(force=False)
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


Skipped full run (RUN_FULL_RUNS=False).


In [17]:
if RUN_FULL_RUNS:
    all_coco = BENCH.run_all_coco(force=False)
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


Skipped full run (RUN_FULL_RUNS=False).


## Leaderboard

One row per (method, suite), ranked by overall accuracy. Read `Voicing False Alarm` alongside it: the synthetic suites have effectively digital silence in their unvoiced frames, while original Bach10 and URMP supply real isolated recordings. Neither substitutes for a live microphone benchmark in a real room.

With `RUN_FULL_RUNS = False`, this table uses the three preliminary suites.

In [18]:
leaderboard = BENCH.show_leaderboard(
    *(
        (all_audio, all_urmp, all_coco)
        if RUN_FULL_RUNS
        else (preliminary, preliminary_urmp, preliminary_coco)
    )
)


,,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,suite,,,,,,,
attune,preliminary_coco,26,0.9612,0.9636,0.9557,0.9798,0.1434,7.8474
crepe,preliminary_coco,26,0.9617,0.9633,0.9507,0.9832,0.2489,0.1825
pyin,preliminary_coco,26,0.9648,0.9673,0.9500,0.9868,0.3191,6.1414
basic_pitch,preliminary_coco,26,0.9658,0.9663,0.9462,0.9959,0.4086,37.5741
praat,preliminary_coco,26,0.9474,0.9575,0.9423,0.9836,0.1491,679.2098
attune,preliminary_urmp,26,0.9455,0.9516,0.9328,0.9699,0.1087,8.2887
praat,preliminary_bach10_original,8,0.9579,0.9629,0.9312,0.9787,0.2763,273.7342
crepe,preliminary_bach10_original,8,0.9717,0.9723,0.9302,0.9809,0.3930,0.1887
attune,preliminary_bach10_original,8,0.9596,0.9602,0.9287,0.9671,0.3121,8.2137


## Terminal equivalents

The same suites from a shell, for long overnight runs. `--write` publishes clean-suite frame-pooled metrics to `results/pitch/pitch_frame_weighted_benchmarks.csv`. Per-track results are written under `results/pitch/raw_outputs/<method>/`.

```shell
# balanced preliminary Bach10 draw, plus dependency availability
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets bach10-original --per-instrument 2 --seed 0 --list

# full original Bach10 plus MDB synthesized corpora
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets mdb-stem-synth mdb-melody-synth bach10-original --write

# balanced preliminary URMP draw, then the complete 149-stem corpus
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets urmp --per-instrument 2 --seed 0
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets urmp --write

# balanced preliminary CocoChorales draw
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets coco --split test --per-instrument 2 --seed 0 --materialize --write

# the heavy PyTorch models in their own invocation, so runtimes do not stack
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets bach10-original --methods torchcrepe penn --write

# resume an interrupted run, or re-score existing caches without detecting
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets coco --skip-cached --write
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets coco --cache-only --write
```